In [1]:
# Comparación de Modelos Predictivos y Entrenamiento de modelos
## Random Forest, Gradient Boosting, XGBoost

In [2]:
import sys
!{sys.executable} -m pip install xgboost

Defaulting to user installation because normal site-packages is not writeable


In [3]:
import seaborn as sns

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)


In [5]:
# Cargar datos
df = pd.read_excel('../data/raw/dataset_sintetico_demanda_lima.xlsx')
df['fecha'] = pd.to_datetime(df['fecha'])
df = df.sort_values('fecha').reset_index(drop=True)

# Features
df['dia_semana'] = df['fecha'].dt.dayofweek
df['mes'] = df['fecha'].dt.month
df['dia'] = df['fecha'].dt.day

for lag in [1, 2, 3, 7]:
    df[f'lag_{lag}'] = df['demanda_real'].shift(lag)

df['media_movil_7'] = df['demanda_real'].rolling(7).mean()
df = df.dropna().reset_index(drop=True)

feature_cols = ['dia_semana', 'mes', 'dia', 'lag_1', 'lag_2', 'lag_3', 'lag_7', 'media_movil_7']
X = df[feature_cols].values
y = df['demanda_real'].values

# Dividir datos (80% entrenamiento, 20% prueba)
split_idx = int(len(X) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f"Datos: {len(df)} días")
print(f"Train: {len(X_train)} días, Test: {len(X_test)} días")
print(f"Features: {len(feature_cols)}")

Datos: 723 días
Train: 578 días, Test: 145 días
Features: 8


In [6]:
# ============================================
# FUNCIÓN PARA CALCULAR LAS 5 MÉTRICAS
# ============================================
def calcular_metricas_completas(y_true, y_pred, X_test):
    """
    Calcula MAE, RMSE, MAPE, R² y R² Ajustado
    """
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    r2 = r2_score(y_true, y_pred)
    
    # R² Ajustado
    n = len(y_true)
    p = X_test.shape[1]
    r2_ajustado = 1 - (1 - r2) * (n - 1) / (n - p - 1)
    
    return {
        'MAE': round(mae, 2),
        'RMSE': round(rmse, 2),
        'MAPE': round(mape, 2),
        'R2': round(r2, 4),
        'R2_Ajustado': round(r2_ajustado, 4)
    }

In [7]:
# Preparar features
df['dia_semana'] = df['fecha'].dt.dayofweek
df['mes'] = df['fecha'].dt.month
df['dia'] = df['fecha'].dt.day

for lag in [1, 2, 3, 7]:
    df[f'lag_{lag}'] = df['demanda_real'].shift(lag)

df['media_movil_7'] = df['demanda_real'].rolling(7).mean()
df = df.dropna().reset_index(drop=True)

feature_cols = ['dia_semana', 'mes', 'dia', 'lag_1', 'lag_2', 'lag_3', 'lag_7', 'media_movil_7']
X = df[feature_cols].values
y = df['demanda_real'].values

# División train/test (80/20 respetando orden)
split_idx = int(len(X) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f"Train: {len(X_train)} días")
print(f"Test: {len(X_test)} días")

Train: 572 días
Test: 144 días


In [8]:
# ============================================
# ENTRENAMIENTO DE LOS 3 MODELOS
# ============================================
print("\n" + "=" * 60)
print(" ENTRENANDO MODELOS (Random Forest, Gradient Boosting, XGBoost)")
print("=" * 60)

resultados = []


 ENTRENANDO MODELOS (Random Forest, Gradient Boosting, XGBoost)


In [9]:
# 1. RANDOM FOREST
print("=" * 60)
print("1. RANDOM FOREST")
print("=" * 60)

rf = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
metrics_rf = calcular_metricas_completas(y_test, y_pred_rf, X_test)
metrics_rf['Modelo'] = 'Random Forest'

print(f"MAE: {metrics_rf['MAE']} | RMSE: {metrics_rf['RMSE']} | MAPE: {metrics_rf['MAPE']}%")
print(f"R²: {metrics_rf['R2']} | R² Ajustado: {metrics_rf['R2_Ajustado']}")

1. RANDOM FOREST
MAE: 23.96 | RMSE: 30.72 | MAPE: 12.9%
R²: 0.4762 | R² Ajustado: 0.4452


In [10]:
# 2. GRADIENT BOOSTING
print("=" * 60)
print("2. GRADIENT BOOSTING")
print("=" * 60)

gb = GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=5, random_state=42)
gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)
metrics_gb = calcular_metricas_completas(y_test, y_pred_gb, X_test)
metrics_gb['Modelo'] = 'Gradient Boosting'

print(f"MAE: {metrics_gb['MAE']} | RMSE: {metrics_gb['RMSE']} | MAPE: {metrics_gb['MAPE']}%")
print(f"R²: {metrics_gb['R2']} | R² Ajustado: {metrics_gb['R2_Ajustado']}")

2. GRADIENT BOOSTING
MAE: 25.07 | RMSE: 31.63 | MAPE: 13.2%
R²: 0.4448 | R² Ajustado: 0.4119


In [11]:
# 3. XGBOOST
print("=" * 60)
print("3. XGBOOST")
print("=" * 60)

xgb = XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=6, random_state=42, verbosity=0)
xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)
metrics_xgb = calcular_metricas_completas(y_test, y_pred_xgb, X_test)
metrics_xgb['Modelo'] = 'XGBoost'

print(f"MAE: {metrics_xgb['MAE']} | RMSE: {metrics_xgb['RMSE']} | MAPE: {metrics_xgb['MAPE']}%")
print(f"R²: {metrics_xgb['R2']} | R² Ajustado: {metrics_xgb['R2_Ajustado']}")

3. XGBOOST
MAE: 25.2 | RMSE: 32.1 | MAPE: 13.25%
R²: 0.4281 | R² Ajustado: 0.3943


In [12]:
# Crear DataFrame con resultados
resultados = [metrics_rf, metrics_gb, metrics_xgb]
df_resultados = pd.DataFrame(resultados)
df_resultados = df_resultados[['Modelo', 'MAE', 'RMSE', 'MAPE', 'R2', 'R2_Ajustado']]
df_resultados = df_resultados.sort_values('MAE')

print("\n" + "=" * 70)
print("TABLA COMPARATIVA DE MODELOS")
print("=" * 70)
print(df_resultados.to_string(index=False))


TABLA COMPARATIVA DE MODELOS
           Modelo   MAE  RMSE  MAPE     R2  R2_Ajustado
    Random Forest 23.96 30.72 12.90 0.4762       0.4452
Gradient Boosting 25.07 31.63 13.20 0.4448       0.4119
          XGBoost 25.20 32.10 13.25 0.4281       0.3943


In [14]:
# ============================================
# Guardar tabla de resultados
# ============================================
import os
import joblib

# Crear carpetas si no existen
os.makedirs('../reports/tables', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

# Guardar tabla comparativa
df_resultados.to_csv('../reports/tables/model_comparison_full.csv', index=False)

print("\nResultados guardados en reports/tables/model_comparison_full.csv")

# ============================================
# Guardar modelos y datos para Notebook 03
# ============================================

modelos = {
    'random_forest': {
        'modelo': rf,
        'y_pred': y_pred_rf,
        'y_test': y_test,
        'X_test': X_test
    },
    'gradient_boosting': {
        'modelo': gb,
        'y_pred': y_pred_gb,
        'y_test': y_test,
        'X_test': X_test
    },
    'xgboost': {
        'modelo': xgb,
        'y_pred': y_pred_xgb,
        'y_test': y_test,
        'X_test': X_test
    }
}

for nombre, datos in modelos.items():
    joblib.dump(
        {
            'modelo': datos['modelo'],
            'y_pred': datos['y_pred'],
            'y_test': datos['y_test'],
            'X_train': X_train,
            'X_test': datos['X_test'],
            'y_train': y_train
        },
        f'../data/processed/{nombre}_data.pkl'
    )

print("\nModelos guardados en data/processed/")


Resultados guardados en reports/tables/model_comparison_full.csv

Modelos guardados en data/processed/
